In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → 다음 셀부터
# (google-colab·numba·cudf 버전 경고와 bottleneck 경고는 무시해도 된다)
# 로컬은 .venv에 requirements.txt로 이미 설치되어 있으면 건너뛴다: python -m pip install -r requirements.txt
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 런타임 → 세션 다시 시작 후 다음 셀부터 실행')
else:
    print('로컬: 설치 생략 (requirements.txt 버전 사용)')

# 팀 실험 예시

**자유롭게 실험하고, 결과 표만 `outputs/model_results/<내이름>/`에 올립니다.** 이 노트북은 예시일 뿐이며 형식을 따를 필요는 없습니다.

**지킬 것 3가지**
1. 2022~2025년(테스트) 자료는 열지 않는다. `data/raw/test_2022_2025__*.csv`를 열거나 `data/` 폴더째 검색·복사하지 않는다.
2. 채점은 모두 같게 한다.
   - 기간: 2006-01 ~ 2021-12, 192개월
   - 대상: 매달 다음 달 수익률이 있는 전 종목(133,876행). 특징이 비면 채워서라도 예측한다.
   - 점수: 매달 예측과 실제 수익률의 Spearman 순위 상관(IC) → 192개월 평균 IC, ICIR(평균 ÷ 표준편차)
3. 미래 정보 금지. 어떤 해를 예측하는 모델은 그 전 해까지의 자료로만 학습하고 하이퍼파라미터를 고른다.

**나머지는 자유**: 모델, 특징, 하이퍼파라미터, 학습 기간, 노트북 형식.

**결과**: 표(csv)를 `outputs/model_results/<내이름>/`에 넣고 커밋·푸시·PR. 칸은 모델, 하이퍼파라미터, 조건(바꾼 것), 평균 IC, ICIR. 예측값 파일(parquet)은 커서 올리지 않는다.

이 노트북은 복사해서 `notebook/<내이름>/`에서 쓰세요(원본을 고치면 Git 충돌).

## 1. 설정

- `AUTHOR`: 내 이름(영문). 결과 표가 `outputs/model_results/<AUTHOR>/`에 저장됩니다.
- `FEATURES`: 새 특징 파일이 있으면 `{이름: 'data/processed/<파일>.parquet'}` (열: `stock_id`, `base_month`(YYYY-MM), `<이름>`), 없으면 `{}`.
- **Colab**: 공유 폴더 `금융데이터인텔리전스`를 내 드라이브에 바로가기로 추가(이름 그대로), 런타임 GPU, 맨 위 설치 셀 → 런타임 다시 시작 → 이 셀. `GIT_REF`에 브랜치 이름(예: `'main'`). 결과 표는 내 드라이브 `model_results/<AUTHOR>/`에 생기니 내려받아 로컬 같은 경로에 넣고 커밋합니다.

In [ ]:
import os, sys, shutil, subprocess, time
from pathlib import Path

AUTHOR = ''        # 예: 'hong'
FEATURES = {}      # 예: {'h2': 'data/processed/h2.parquet'}
GIT_REF = ''       # Colab에서만: 브랜치 이름(예: 'main')

IN_COLAB = 'google.colab' in sys.modules
if not AUTHOR:
    raise ValueError('AUTHOR에 내 이름(영문)을 적는다.')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise ValueError("GIT_REF에 브랜치 이름(예: 'main')을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/JiunShin1214/finance_analytics.git', str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    if not (ROOT / 'data').is_symlink():          # 저장소 data/를 Drive 공유 data로 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(Path('/content/drive/MyDrive/금융데이터인텔리전스/data'))
    OUT_DIR = Path('/content/drive/MyDrive/model_results') / AUTHOR
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_DIR = ROOT / 'outputs' / 'model_results' / AUTHOR
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
GPU_NAME = (subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
            if shutil.which('nvidia-smi') else None)
N_THREADS = os.cpu_count()
OUT_DIR.mkdir(parents=True, exist_ok=True)
ROWS = []          # 결과 표: 실험마다 한 줄씩 쌓는다
pd.set_option('display.width', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT:', ROOT, '| 결과 표 위치:', OUT_DIR, '| GPU:', GPU_NAME)

## 2. 입력 준비 (약 2분)

`data/`의 공통 입력(H1·σ·목표값)을 종목 × 월 표로 만듭니다.
- `data`: 입력 열 `x_size`, `x_bm`, `x_mom`, `x_sigma_span<개월>`(매달 순위 [−1, 1]), 새 특징은 `x_<이름>` / 목표값 열 `t_raw`, `t_winsor_1_99`, `t_monthly_zscore`, `t_percentile`, `t_decile` / 원수익률 `y`, 연도 `year`, 목표월 `target_month`
- `val`: 채점할 133,876행(2006~2021, 수익률 있는 전 종목)과 순서

In [ ]:
prepared = mt.prepare_inputs(ROOT, config, extra_features=FEATURES or None)
data = prepared['data']
val = mt.validation_rows(prepared, config)
FOLDS = mt.plan_folds(config['MODEL_GRID'])     # 채점 연도 2006~2021, 해마다 하나
print('채점 행', len(val), '| 채점 연도', FOLDS[0]['valid_year'], '~', FOLDS[-1]['valid_year'])

## 3. 채점 함수

예측(`val` 순서, 133,876개)을 넣으면 192개월 IC를 계산해 평균 IC·ICIR·IC>0 비율을 돌려줍니다. 한 달 예측이 모두 같으면 그 달 IC는 0으로 셉니다(우리 결과와 같은 계산).

In [ ]:
def score(pred):
    t = mt.monthly_ic(val.target_month.to_numpy(), np.asarray(pred, float), val.y.to_numpy(float))
    ic = t.ic.where(t.reason.ne('constant_score'), 0.0)
    return {'months': int(ic.notna().sum()), 'mean_ic': ic.mean(), 'icir': ic.mean() / ic.std(), 'ic_pos': (ic > 0).mean()}

## 4. 예시 A — 공용 부스팅에 하이퍼파라미터 바꿔 돌리기

- `SETTINGS`: (종류, 목표값, 손실, 깊이, 학습률). 종류별 목표값·손실은 `lgb_reg`·`xgb_reg`: `winsor_1_99`·`monthly_zscore` × `mse`·`huber_sigma135` / `lgb_rankreg`·`xgb_rankreg`: `percentile`·`mse` / `xgb_pairwise`: `decile`·`pairwise`
- `OVERRIDES`: 라이브러리 설정. L2는 LightGBM `lambda_l2`, XGBoost `lambda` / 잎 최소 행 `min_data_in_leaf` / 행 추출 `bagging_fraction`(+`bagging_freq: 1`)·`subsample` / 열 추출 `feature_fraction`·`colsample_bytree`
- 트리 1000개 중 직전 1년(내부 검증) IC가 가장 높은 트리 수, σ 기간(12·36·60개월)을 해마다 고릅니다.
- LightGBM은 로컬 CPU(설정 1개당 약 6분), XGBoost는 Colab GPU.

In [ ]:
KIND = 'lgb_rankreg'
SETTINGS = [(KIND, 'percentile', 'mse', 3, 0.01)]
INPUTS = ['size', 'bm', 'mom', 'sigma']
OVERRIDES = {'lambda_l2': 5.0}

t0 = time.perf_counter()
res = mt.run_boosting(prepared, config, KIND, phase='individual', settings=SETTINGS, sets=dict([mt.make_input_set(INPUTS)]),
                      overrides=OVERRIDES, n_threads=N_THREADS)
minutes = (time.perf_counter() - t0) / 60
for cid, pred in res['predictions'].items():
    c = mt.split_config_id(cid)
    ROWS.append({'author': AUTHOR, 'model': f"{c['model']} {c['target']} {c['loss']}", 'hparams': c['hparams'],
                 'inputs': '+'.join(INPUTS), 'conditions': '기본', **score(pred),
                 'train_ic': res['selections'].train_ic.mean(), 'minutes': round(minutes, 1)})
pd.DataFrame(ROWS)

## 5. 예시 B — 직접 만든 모델

해마다(Fold마다) 그 전 해까지의 행으로 학습하고 그 해를 예측합니다. `mt.fold_rows`는 기본 행을 줍니다: `fit`(학습), `inner`(직전 1년, 하이퍼파라미터 고르기용), `out`(채점). 바꿔 써도 됩니다(미래 자료만 쓰지 않으면 됨).

아래는 OLS 예시입니다. 내 모델 코드는 `src/team/<내이름>/`에 두고 불러 써도 됩니다.

In [ ]:
from sklearn.linear_model import LinearRegression

COLS = ['x_size', 'x_bm', 'x_mom', 'x_sigma_span36']
TARGET = 't_monthly_zscore'
pred = np.full(len(val), np.nan)
train_ics = []
for f in FOLDS:
    m = mt.fold_rows(prepared, f)
    X, y = data.loc[m['fit'], COLS].to_numpy(float), data.loc[m['fit'], TARGET].to_numpy(float)
    model = LinearRegression().fit(X, y)
    pred[val.fold.eq(f['fold']).to_numpy()] = model.predict(data.loc[m['out'], COLS].fillna(0).to_numpy(float))   # 빈 특징은 0
    train_ics.append(mt.mean_ic(mt.monthly_ic(data.target_month.to_numpy()[m['fit']], model.predict(X), data.y.to_numpy(float)[m['fit']])))
ROWS.append({'author': AUTHOR, 'model': 'OLS monthly_zscore mse', 'hparams': '없음', 'inputs': 'size+bm+mom+sigma(span36)',
             'conditions': '기본', **score(pred), 'train_ic': np.mean(train_ics)})
pd.DataFrame(ROWS)

## 6. 결과 표 저장

`outputs/model_results/<AUTHOR>/<파일 이름>.csv`로 저장합니다. 그다음 `git add outputs/model_results/<AUTHOR>` → 커밋 → 푸시 → PR.

In [ ]:
NAME = 'example'     # 파일 이름(자유)
pd.DataFrame(ROWS).to_csv(OUT_DIR / f'{NAME}.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT_DIR / f'{NAME}.csv')